# QaptaanLM-0.75B-Instruct: SFT Hub Publisher
### Package & Publish SFT Model (Checkpoint-12208) to Hugging Face Hub

This self-contained notebook:
1. Locates the JAX SFT checkpoint (`checkpoint-12208` 100M tokens) from the attached `kaptaan45/checkpoints-sft` dataset.
2. Restores JAX PyTree and converts Flax parameters to PyTorch `model.safetensors` (enforcing tied word embeddings for exact **752M parameters**, and 3D Conv1D weights `[6144, 1, 4]`).
3. Embeds custom architecture (`configuration_qaptaan.py` and `modeling_qaptaan.py`), `config.json`, `generation_config.json`, and official Qwen3.5 tokenizer.
4. Validates local PyTorch loading with `AutoModelForCausalLM` and runs test code generation.
5. Uploads the complete model card and repository to **`kaptaan45/QaptaanLM-0.75B-Instruct`** on Hugging Face Hub!

## 1. Environment & Dependencies Setup

In [ ]:
!pip uninstall -y torchvision torchaudio
!pip install -q --upgrade scipy transformers accelerate safetensors huggingface_hub orbax-checkpoint flax jax kaggle


## 2. Authenticate with Hugging Face & Kaggle

In [ ]:
import os, json
from huggingface_hub import HfApi, login, create_repo

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = input('Enter your Hugging Face write token (hf_...): ').strip()

login(token=HF_TOKEN, add_to_git_credential=True)
print('✓ Authenticated with Hugging Face successfully!')

# Configure Kaggle credentials
KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', 'kaptaan45')
KAGGLE_KEY = os.environ.get('KAGGLE_KEY', '')
try:
    from kaggle_secrets import UserSecretsClient
    KAGGLE_KEY = UserSecretsClient().get_secret('KAGGLE_KEY') or KAGGLE_KEY
except Exception:
    pass
if KAGGLE_KEY:
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY
    print(f'✓ Kaggle credentials active for user: {KAGGLE_USER}')


## 3. Restore JAX SFT Checkpoint & Convert to 752M PyTorch Safetensors

In [ ]:
import glob, re, shutil
from pathlib import Path
import numpy as np
import orbax.checkpoint as ocp
from safetensors.numpy import save_file as save_numpy_safetensors

export_dir = Path('/kaggle/working/qaptaanlm_instruct_hf') if Path('/kaggle/working').exists() else Path('./qaptaanlm_instruct_hf')
export_dir.mkdir(parents=True, exist_ok=True)

# Locate SFT Checkpoint
ckpts = glob.glob('/kaggle/input/**/checkpoint-*', recursive=True) or glob.glob('./**/*checkpoint-*', recursive=True)
if not ckpts:
    print('Checkpoint not mounted in /kaggle/input. Downloading dataset kaptaan45/checkpoints-sft via Kaggle API...')
    import kaggle
    dl_path = Path('/kaggle/working/ckpts_download')
    dl_path.mkdir(parents=True, exist_ok=True)
    kaggle.api.dataset_download_files('kaptaan45/checkpoints-sft', path=str(dl_path), unzip=True)
    ckpts = glob.glob(str(dl_path / '**' / 'checkpoint-*'), recursive=True)

print(f'Found {len(ckpts)} checkpoint directories:', ckpts)
selected_ckpt = sorted(ckpts, key=lambda x: int(re.findall(r'checkpoint-(\d+)', x)[-1]) if re.findall(r'checkpoint-(\d+)', x) else 0)[-1]
print(f'✓ Selected final checkpoint: {selected_ckpt}')

state_path = Path(selected_ckpt) / 'state' if (Path(selected_ckpt) / 'state').exists() else Path(selected_ckpt)
print(f'Restoring JAX PyTree from {state_path}...')
checkpointer = ocp.StandardCheckpointer()
restored = checkpointer.restore(state_path)
params = restored['params']
print(f'✓ Restored {len(params["model"])} model parameter modules!')

# Flax -> PyTorch conversion logic
def convert_flax_params_to_pytorch(model_params, num_layers=24):
    sd = {}
    def to_np(arr):
        return np.array(arr)
    
    if 'embed_tokens' in model_params:
        sd['model.embed_tokens.weight'] = to_np(model_params['embed_tokens']['embedding'])
    if 'norm' in model_params:
        sd['model.norm.weight'] = to_np(model_params['norm']['weight'])
        
    for i in range(num_layers):
        layer_name = f'layers_{i}'
        if layer_name not in model_params:
            continue
        ld = model_params[layer_name]
        prefix = f'model.layers.{i}.'
        
        if 'input_layernorm' in ld:
            sd[f'{prefix}input_layernorm.weight'] = to_np(ld['input_layernorm']['weight'])
        if 'post_attention_layernorm' in ld:
            sd[f'{prefix}post_attention_layernorm.weight'] = to_np(ld['post_attention_layernorm']['weight'])
        if 'mlp' in ld:
            sd[f'{prefix}mlp.gate_proj.weight'] = to_np(ld['mlp']['gate_proj']['kernel'].T)
            sd[f'{prefix}mlp.up_proj.weight'] = to_np(ld['mlp']['up_proj']['kernel'].T)
            sd[f'{prefix}mlp.down_proj.weight'] = to_np(ld['mlp']['down_proj']['kernel'].T)
        if 'self_attn' in ld:
            sa = ld['self_attn']
            sd[f'{prefix}self_attn.q_proj.weight'] = to_np(sa['q_proj']['kernel'].T)
            sd[f'{prefix}self_attn.k_proj.weight'] = to_np(sa['k_proj']['kernel'].T)
            sd[f'{prefix}self_attn.v_proj.weight'] = to_np(sa['v_proj']['kernel'].T)
            sd[f'{prefix}self_attn.o_proj.weight'] = to_np(sa['o_proj']['kernel'].T)
            sd[f'{prefix}self_attn.q_norm.weight'] = to_np(sa['q_norm']['weight'])
            sd[f'{prefix}self_attn.k_norm.weight'] = to_np(sa['k_norm']['weight'])
        if 'linear_attn' in ld:
            la = ld['linear_attn']
            sd[f'{prefix}linear_attn.in_proj_qkv.weight'] = to_np(la['in_proj_qkv']['kernel'].T)
            sd[f'{prefix}linear_attn.in_proj_z.weight'] = to_np(la['in_proj_z']['kernel'].T)
            sd[f'{prefix}linear_attn.in_proj_b.weight'] = to_np(la['in_proj_b']['kernel'].T)
            sd[f'{prefix}linear_attn.in_proj_a.weight'] = to_np(la['in_proj_a']['kernel'].T)
            sd[f'{prefix}linear_attn.out_proj.weight'] = to_np(la['out_proj']['kernel'].T)
            sd[f'{prefix}linear_attn.norm.weight'] = to_np(la['norm']['weight'])
            sd[f'{prefix}linear_attn.dt_bias'] = to_np(la['dt_bias'])
            sd[f'{prefix}linear_attn.A_log'] = to_np(la['A_log'])
            conv_w = to_np(la['conv1d_weight'])
            if conv_w.ndim == 2:
                conv_w = conv_w.reshape(conv_w.shape[0], 1, conv_w.shape[1])
            sd[f'{prefix}linear_attn.conv1d.weight'] = conv_w
    return sd

print('Converting parameters to PyTorch state dict...')
state_dict = convert_flax_params_to_pytorch(params['model'])
total_params = sum(v.size for v in state_dict.values())
conv_count = sum(1 for k in state_dict if 'conv1d.weight' in k)
print(f'✓ Converted {len(state_dict)} tensors: {total_params:,} parameters ({total_params/1e6:.2f}M)')
print(f'✓ Validated {conv_count} Conv1D linear attention kernel tensors')
assert 750_000_000 <= total_params <= 755_000_000, f'Unexpected param count: {total_params}'

st_path = export_dir / 'model.safetensors'
print(f'Saving clean 752M safetensors to {st_path}...')
save_numpy_safetensors(state_dict, str(st_path))
print(f'✓ Saved model.safetensors ({st_path.stat().st_size / (1024*1024):.2f} MB)')


## 4. Package Modeling, Configuration, Tokenizer & Model Card

In [ ]:
# 1. configuration_qaptaan.py
config_code = '"""QaptaanLM-0.75B Configuration."""\n\nfrom transformers.configuration_utils import PretrainedConfig\n\n\nclass QaptaanConfig(PretrainedConfig):\n    model_type = "qaptaan"\n    keys_to_ignore_at_inference = ["past_key_values"]\n\n    def __init__(\n        self,\n        vocab_size: int = 248320,\n        hidden_size: int = 1024,\n        intermediate_size: int = 3584,\n        num_hidden_layers: int = 24,\n        num_attention_heads: int = 8,\n        num_key_value_heads: int = 2,\n        head_dim: int = 256,\n        rms_norm_eps: float = 1e-6,\n        tie_word_embeddings: bool = True,\n        max_position_embeddings: int = 262144,\n        rope_theta: float = 10000000.0,\n        partial_rotary_factor: float = 0.25,\n        attn_output_gate: bool = True,\n        full_attention_interval: int = 4,\n        linear_key_head_dim: int = 128,\n        linear_value_head_dim: int = 128,\n        linear_num_key_heads: int = 16,\n        linear_num_value_heads: int = 16,\n        linear_conv_kernel_dim: int = 4,\n        hidden_act: str = "silu",\n        initializer_range: float = 0.02,\n        use_cache: bool = True,\n        bos_token_id: int = None,\n        eos_token_id: int = 248044,\n        pad_token_id: int = 248044,\n        **kwargs,\n    ):\n        self.vocab_size = vocab_size\n        self.hidden_size = hidden_size\n        self.intermediate_size = intermediate_size\n        self.num_hidden_layers = num_hidden_layers\n        self.num_attention_heads = num_attention_heads\n        self.num_key_value_heads = num_key_value_heads\n        self.head_dim = head_dim\n        self.rms_norm_eps = rms_norm_eps\n        self.tie_word_embeddings = tie_word_embeddings\n        self.max_position_embeddings = max_position_embeddings\n        self.rope_theta = rope_theta\n        self.partial_rotary_factor = partial_rotary_factor\n        self.attn_output_gate = attn_output_gate\n        self.full_attention_interval = full_attention_interval\n        self.linear_key_head_dim = linear_key_head_dim\n        self.linear_value_head_dim = linear_value_head_dim\n        self.linear_num_key_heads = linear_num_key_heads\n        self.linear_num_value_heads = linear_num_value_heads\n        self.linear_conv_kernel_dim = linear_conv_kernel_dim\n        self.hidden_act = hidden_act\n        self.initializer_range = initializer_range\n        self.use_cache = use_cache\n\n        # Auto-compute layer types (hybrid 3:1 linear-to-full attention)\n        self.layer_types = []\n        for i in range(num_hidden_layers):\n            if (i + 1) % full_attention_interval == 0:\n                self.layer_types.append("full_attention")\n            else:\n                self.layer_types.append("linear_attention")\n\n        super().__init__(\n            bos_token_id=bos_token_id,\n            eos_token_id=eos_token_id,\n            pad_token_id=pad_token_id,\n            tie_word_embeddings=tie_word_embeddings,\n            **kwargs,\n        )\n'
with open(export_dir / 'configuration_qaptaan.py', 'w', encoding='utf-8') as f:
    f.write(config_code)
print('✓ Created configuration_qaptaan.py')

# 2. modeling_qaptaan.py
modeling_code = '"""QaptaanLM-0.75B PyTorch Model Implementation with Exact JAX Recurrence & Fast O(1) Cache."""\n\nimport math\nfrom typing import Any, Dict, List, Optional, Tuple, Union\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers.modeling_outputs import CausalLMOutputWithPast, BaseModelOutputWithPast\nfrom transformers.modeling_utils import PreTrainedModel\nfrom transformers.generation import GenerationMixin\nfrom transformers.cache_utils import Cache\n\nfrom .configuration_qaptaan import QaptaanConfig\n\n\nclass QaptaanCache:\n    """Hybrid State Cache storing Conv1D state, Gated Delta Net recurrent state, and Full Attention KV cache."""\n\n    def __init__(self):\n        self.conv_states: Dict[int, torch.Tensor] = {}\n        self.recurrent_states: Dict[int, torch.Tensor] = {}\n        self.key_cache: Dict[int, torch.Tensor] = {}\n        self.value_cache: Dict[int, torch.Tensor] = {}\n        self._seen_tokens: int = 0\n\n    def get_seq_length(self, layer_idx: Optional[int] = 0) -> int:\n        return self._seen_tokens\n\n    def update(\n        self,\n        key_states: torch.Tensor,\n        value_states: torch.Tensor,\n        layer_idx: int,\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        if layer_idx not in self.key_cache:\n            self.key_cache[layer_idx] = key_states\n            self.value_cache[layer_idx] = value_states\n        else:\n            self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], key_states], dim=2)\n            self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], value_states], dim=2)\n        return self.key_cache[layer_idx], self.value_cache[layer_idx]\n\n\nclass QaptaanRMSNorm(nn.Module):\n    def __init__(self, dim: int, eps: float = 1e-6):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(dim))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        variance = x.pow(2).mean(-1, keepdim=True)\n        normed = x * torch.rsqrt(variance + self.eps)\n        return normed * self.weight\n\n\nclass QaptaanRMSNormGated(nn.Module):\n    def __init__(self, dim: int, eps: float = 1e-6):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(dim))\n\n    def forward(self, x: torch.Tensor, gate: torch.Tensor) -> torch.Tensor:\n        variance = x.pow(2).mean(-1, keepdim=True)\n        normed = x * torch.rsqrt(variance + self.eps)\n        normed = normed * self.weight\n        return normed * F.silu(gate)\n\n\nclass QaptaanRotaryEmbedding(nn.Module):\n    def __init__(self, dim: int, max_position_embeddings: int = 262144, base: float = 10000000.0):\n        super().__init__()\n        self.dim = dim\n        self.max_position_embeddings = max_position_embeddings\n        self.base = base\n        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2, dtype=torch.float32) / self.dim))\n        self.register_buffer("inv_freq", inv_freq, persistent=False)\n\n    def forward(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:\n        t = torch.arange(seq_len, device=device, dtype=torch.float32)\n        freqs = torch.outer(t, self.inv_freq)\n        emb = torch.cat([freqs, freqs], dim=-1)\n        return emb.cos().to(dtype), emb.sin().to(dtype)\n\n\ndef rotate_half(x: torch.Tensor) -> torch.Tensor:\n    x1 = x[..., : x.shape[-1] // 2]\n    x2 = x[..., x.shape[-1] // 2 :]\n    return torch.cat([-x2, x1], dim=-1)\n\n\ndef apply_rope(\n    query: torch.Tensor, key: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, rotary_dim: int\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    q_rot, q_pass = query[..., :rotary_dim], query[..., rotary_dim:]\n    k_rot, k_pass = key[..., :rotary_dim], key[..., rotary_dim:]\n\n    cos = cos.unsqueeze(0).unsqueeze(0)  # [1, 1, S, rotary_dim]\n    sin = sin.unsqueeze(0).unsqueeze(0)\n\n    q_rot_embed = (q_rot * cos) + (rotate_half(q_rot) * sin)\n    k_rot_embed = (k_rot * cos) + (rotate_half(k_rot) * sin)\n\n    q_out = torch.cat([q_rot_embed, q_pass], dim=-1)\n    k_out = torch.cat([k_rot_embed, k_pass], dim=-1)\n    return q_out, k_out\n\n\nclass QaptaanMLP(nn.Module):\n    def __init__(self, config: QaptaanConfig):\n        super().__init__()\n        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)\n        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)\n        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))\n\n\nclass QaptaanFullAttention(nn.Module):\n    def __init__(self, config: QaptaanConfig, layer_idx: int):\n        super().__init__()\n        self.config = config\n        self.layer_idx = layer_idx\n        self.head_dim = config.head_dim\n        self.num_heads = config.num_attention_heads\n        self.num_kv_heads = config.num_key_value_heads\n        self.num_kv_groups = self.num_heads // self.num_kv_heads\n        self.scaling = 1.0 / math.sqrt(self.head_dim)\n        self.rotary_dim = int(self.head_dim * config.partial_rotary_factor)\n\n        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim * 2, bias=False)\n        self.k_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)\n        self.v_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=False)\n        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=False)\n\n        self.q_norm = QaptaanRMSNorm(self.head_dim, eps=config.rms_norm_eps)\n        self.k_norm = QaptaanRMSNorm(self.head_dim, eps=config.rms_norm_eps)\n        self.rotary = QaptaanRotaryEmbedding(self.rotary_dim, config.max_position_embeddings, config.rope_theta)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        past_key_value: Optional[QaptaanCache] = None,\n        use_cache: bool = False,\n    ) -> torch.Tensor:\n        batch_size, seq_len, _ = hidden_states.shape\n\n        q_proj_out = self.q_proj(hidden_states)\n        q_proj_out = q_proj_out.view(batch_size, seq_len, self.num_heads, 2 * self.head_dim)\n        query = q_proj_out[..., : self.head_dim]\n        gate = q_proj_out[..., self.head_dim :]\n\n        key = self.k_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)\n        value = self.v_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)\n\n        query = self.q_norm(query).transpose(1, 2)  # [B, H, S, head_dim]\n        key = self.k_norm(key).transpose(1, 2)      # [B, KV_H, S, head_dim]\n        value = value.transpose(1, 2)               # [B, KV_H, S, head_dim]\n\n        seen_tokens = past_key_value.get_seq_length(self.layer_idx) if (use_cache and past_key_value is not None) else 0\n        cos, sin = self.rotary(seen_tokens + seq_len, hidden_states.device, query.dtype)\n        cos = cos[seen_tokens : seen_tokens + seq_len]\n        sin = sin[seen_tokens : seen_tokens + seq_len]\n        query, key = apply_rope(query, key, cos, sin, self.rotary_dim)\n\n        if use_cache and past_key_value is not None:\n            key, value = past_key_value.update(key, value, self.layer_idx)\n\n        # Expand KV heads for GQA\n        if self.num_kv_groups > 1:\n            key_expanded = key.repeat_interleave(self.num_kv_groups, dim=1)\n            value_expanded = value.repeat_interleave(self.num_kv_groups, dim=1)\n        else:\n            key_expanded = key\n            value_expanded = value\n\n        kv_seq_len = key_expanded.shape[-2]\n        scores = torch.matmul(query, key_expanded.transpose(-1, -2)) * self.scaling\n\n        if seq_len > 1:\n            causal_mask = torch.tril(\n                torch.ones(seq_len, kv_seq_len, dtype=torch.bool, device=hidden_states.device),\n                diagonal=kv_seq_len - seq_len,\n            )\n            scores = scores.masked_fill(~causal_mask, float("-inf"))\n\n        if attention_mask is not None:\n            if attention_mask.dim() == 2:\n                if (attention_mask == 0).any():\n                    scores = scores.masked_fill(attention_mask[:, None, None, :].eq(0), float("-inf"))\n            elif attention_mask.dim() == 4:\n                scores = scores + attention_mask\n\n        attn_weights = F.softmax(scores.float(), dim=-1).to(query.dtype)\n        attn_out = torch.matmul(attn_weights, value_expanded).transpose(1, 2)  # [B, S, H, head_dim]\n        attn_out = attn_out * torch.sigmoid(gate.float()).to(query.dtype)\n        attn_out = attn_out.reshape(batch_size, seq_len, self.num_heads * self.head_dim)\n        return self.o_proj(attn_out)\n\n\nclass QaptaanLinearAttention(nn.Module):\n    """Linear Attention implementing the exact JAX CPT Recurrence Formula with fast O(1) Cache."""\n\n    def __init__(self, config: QaptaanConfig, layer_idx: int):\n        super().__init__()\n        self.config = config\n        self.layer_idx = layer_idx\n        self.num_k_heads = config.linear_num_key_heads\n        self.num_v_heads = config.linear_num_value_heads\n        self.head_k_dim = config.linear_key_head_dim\n        self.head_v_dim = config.linear_value_head_dim\n        self.key_dim = self.num_k_heads * self.head_k_dim\n        self.value_dim = self.num_v_heads * self.head_v_dim\n        self.conv_kernel_size = config.linear_conv_kernel_dim\n        self.conv_dim = self.key_dim * 2 + self.value_dim\n\n        self.in_proj_qkv = nn.Linear(config.hidden_size, self.conv_dim, bias=False)\n        self.in_proj_z = nn.Linear(config.hidden_size, self.value_dim, bias=False)\n        self.in_proj_b = nn.Linear(config.hidden_size, self.num_v_heads, bias=False)\n        self.in_proj_a = nn.Linear(config.hidden_size, self.num_v_heads, bias=False)\n\n        self.conv1d = nn.Conv1d(\n            in_channels=self.conv_dim,\n            out_channels=self.conv_dim,\n            bias=False,\n            kernel_size=self.conv_kernel_size,\n            groups=self.conv_dim,\n            padding=self.conv_kernel_size - 1,\n        )\n\n        self.dt_bias = nn.Parameter(torch.ones(self.num_v_heads))\n        self.A_log = nn.Parameter(torch.zeros(self.num_v_heads))\n\n        self.norm = QaptaanRMSNormGated(self.head_v_dim, eps=config.rms_norm_eps)\n        self.out_proj = nn.Linear(self.value_dim, config.hidden_size, bias=False)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        past_key_value: Optional[QaptaanCache] = None,\n        use_cache: bool = False,\n    ) -> torch.Tensor:\n        batch_size, seq_len, _ = hidden_states.shape\n\n        mixed_qkv = self.in_proj_qkv(hidden_states)\n        z = self.in_proj_z(hidden_states)\n        b = self.in_proj_b(hidden_states)\n        a = self.in_proj_a(hidden_states)\n\n        # Fast O(1) Single-Token Cached Step\n        if use_cache and past_key_value is not None and seq_len == 1 and self.layer_idx in past_key_value.recurrent_states:\n            conv_state = past_key_value.conv_states[self.layer_idx]\n            recurrent_state = past_key_value.recurrent_states[self.layer_idx]\n\n            mixed_qkv_t = mixed_qkv.transpose(1, 2)  # [B, conv_dim, 1]\n            window = torch.cat([conv_state, mixed_qkv_t], dim=-1)  # [B, conv_dim, 4]\n            past_key_value.conv_states[self.layer_idx] = window[:, :, 1:].detach()\n\n            w = self.conv1d.weight.squeeze(1)  # [conv_dim, 4]\n            conv_out = (window * w).sum(dim=-1, keepdim=True)  # [B, conv_dim, 1]\n            mixed_qkv = F.silu(conv_out).transpose(1, 2)  # [B, 1, conv_dim]\n\n            query = mixed_qkv[:, :, : self.key_dim].view(batch_size, 1, self.num_k_heads, self.head_k_dim)\n            key = mixed_qkv[:, :, self.key_dim : 2 * self.key_dim].view(batch_size, 1, self.num_k_heads, self.head_k_dim)\n            value = mixed_qkv[:, :, 2 * self.key_dim :].view(batch_size, 1, self.num_v_heads, self.head_v_dim)\n\n            query = query / (torch.norm(query.float(), dim=-1, keepdim=True) + 1e-6).to(query.dtype)\n            key = key / (torch.norm(key.float(), dim=-1, keepdim=True) + 1e-6).to(key.dtype)\n\n            if self.num_v_heads // self.num_k_heads > 1:\n                ratio = self.num_v_heads // self.num_k_heads\n                query = query.repeat_interleave(ratio, dim=2)\n                key = key.repeat_interleave(ratio, dim=2)\n\n            beta = torch.sigmoid(b.float())\n            g = -torch.exp(self.A_log.float()) * F.softplus(a.float() + self.dt_bias.float())\n\n            scale = 1.0 / math.sqrt(self.head_k_dim)\n            q_i = (query.float() * scale).squeeze(1)   # [B, 16, 128]\n            k_i = key.float().squeeze(1)                # [B, 16, 128]\n            v_i = value.float().squeeze(1)              # [B, 16, 128]\n            b_i = beta.squeeze(1).unsqueeze(-1)         # [B, 16, 1]\n            g_i = g.squeeze(1).unsqueeze(-1)            # [B, 16, 1]\n            decay = torch.exp(g_i).unsqueeze(-1)        # [B, 16, 1, 1]\n\n            # O(1) single-token recurrent update (exact JAX formulation)\n            v_prime = torch.einsum("bhk,bhkd->bhd", k_i, recurrent_state)\n            v_new = (v_i - v_prime) * b_i\n\n            attn_inter = torch.einsum("bhk,bhkd->bhd", q_i, recurrent_state) * torch.exp(g_i)\n            qk_dot = torch.sum(q_i * k_i, dim=-1, keepdim=True)\n            out_i = attn_inter + qk_dot * v_new\n\n            new_state = recurrent_state * decay + torch.einsum("bhk,bhd->bhkd", k_i, v_new)\n            past_key_value.recurrent_states[self.layer_idx] = new_state.detach()\n\n            core_out = out_i.unsqueeze(1).to(hidden_states.dtype)\n            z_reshaped = z.view(batch_size, 1, self.num_v_heads, self.head_v_dim)\n            core_out = self.norm(core_out, z_reshaped)\n            core_out = core_out.reshape(batch_size, 1, self.value_dim)\n            return self.out_proj(core_out)\n\n        # Prefill Mode (Full Sequence Scan)\n        mixed_qkv_t = mixed_qkv.transpose(1, 2)\n        conv_out = self.conv1d(mixed_qkv_t)[:, :, :seq_len].transpose(1, 2)\n        mixed_qkv = F.silu(conv_out)\n\n        query = mixed_qkv[:, :, : self.key_dim].view(batch_size, seq_len, self.num_k_heads, self.head_k_dim)\n        key = mixed_qkv[:, :, self.key_dim : 2 * self.key_dim].view(batch_size, seq_len, self.num_k_heads, self.head_k_dim)\n        value = mixed_qkv[:, :, 2 * self.key_dim :].view(batch_size, seq_len, self.num_v_heads, self.head_v_dim)\n\n        query = query / (torch.norm(query.float(), dim=-1, keepdim=True) + 1e-6).to(query.dtype)\n        key = key / (torch.norm(key.float(), dim=-1, keepdim=True) + 1e-6).to(key.dtype)\n\n        if self.num_v_heads // self.num_k_heads > 1:\n            ratio = self.num_v_heads // self.num_k_heads\n            query = query.repeat_interleave(ratio, dim=2)\n            key = key.repeat_interleave(ratio, dim=2)\n\n        beta = torch.sigmoid(b.float())\n        g = -torch.exp(self.A_log.float()) * F.softplus(a.float() + self.dt_bias.float())\n\n        scale = 1.0 / math.sqrt(self.head_k_dim)\n        q_scaled = query.float() * scale\n        k_fp32 = key.float()\n        v_fp32 = value.float()\n\n        state = torch.zeros(\n            batch_size, self.num_v_heads, self.head_k_dim, self.head_v_dim,\n            device=hidden_states.device, dtype=torch.float32\n        )\n        core_out = torch.zeros(\n            batch_size, seq_len, self.num_v_heads, self.head_v_dim,\n            device=hidden_states.device, dtype=torch.float32\n        )\n\n        for t in range(seq_len):\n            q_i = q_scaled[:, t]\n            k_i = k_fp32[:, t]\n            v_i = v_fp32[:, t]\n            b_i = beta[:, t].unsqueeze(-1)\n            g_i = g[:, t].unsqueeze(-1)\n            decay = torch.exp(g_i).unsqueeze(-1)\n\n            v_prime = torch.einsum("bhk,bhkd->bhd", k_i, state)\n            v_new = (v_i - v_prime) * b_i\n\n            attn_inter = torch.einsum("bhk,bhkd->bhd", q_i, state) * torch.exp(g_i)\n            qk_dot = torch.sum(q_i * k_i, dim=-1, keepdim=True)\n            out_i = attn_inter + qk_dot * v_new\n            core_out[:, t] = out_i\n\n            state = state * decay + torch.einsum("bhk,bhd->bhkd", k_i, v_new)\n\n        if use_cache and past_key_value is not None:\n            if seq_len >= 3:\n                last_3 = mixed_qkv_t[:, :, -3:]\n            else:\n                last_3 = F.pad(mixed_qkv_t, (3 - seq_len, 0))\n            past_key_value.conv_states[self.layer_idx] = last_3.detach()\n            past_key_value.recurrent_states[self.layer_idx] = state.detach()\n\n        core_out = core_out.to(hidden_states.dtype)\n        z_reshaped = z.view(batch_size, seq_len, self.num_v_heads, self.head_v_dim)\n        core_out = self.norm(core_out, z_reshaped)\n        core_out = core_out.reshape(batch_size, seq_len, self.value_dim)\n        return self.out_proj(core_out)\n\n\nclass QaptaanDecoderLayer(nn.Module):\n    def __init__(self, config: QaptaanConfig, layer_idx: int):\n        super().__init__()\n        self.config = config\n        self.layer_idx = layer_idx\n        self.is_full_attention = (layer_idx + 1) % config.full_attention_interval == 0\n\n        self.input_layernorm = QaptaanRMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        if self.is_full_attention:\n            self.self_attn = QaptaanFullAttention(config, layer_idx)\n        else:\n            self.linear_attn = QaptaanLinearAttention(config, layer_idx)\n\n        self.post_attention_layernorm = QaptaanRMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.mlp = QaptaanMLP(config)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        past_key_value: Optional[QaptaanCache] = None,\n        use_cache: bool = False,\n    ) -> torch.Tensor:\n        residual = hidden_states\n        normed = self.input_layernorm(hidden_states)\n\n        if self.is_full_attention:\n            attn_out = self.self_attn(\n                normed, attention_mask=attention_mask, past_key_value=past_key_value, use_cache=use_cache\n            )\n        else:\n            attn_out = self.linear_attn(normed, past_key_value=past_key_value, use_cache=use_cache)\n\n        hidden_states = residual + attn_out\n        residual = hidden_states\n        hidden_states = residual + self.mlp(self.post_attention_layernorm(hidden_states))\n        return hidden_states\n\n\nclass QaptaanPreTrainedModel(PreTrainedModel):\n    config_class = QaptaanConfig\n    base_model_prefix = "model"\n    supports_gradient_checkpointing = False\n    _no_split_modules = ["QaptaanDecoderLayer"]\n\n    def _supports_default_dynamic_cache(self) -> bool:\n        return False\n\n\nclass QaptaanModel(QaptaanPreTrainedModel):\n    def __init__(self, config: QaptaanConfig):\n        super().__init__(config)\n        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)\n        self.layers = nn.ModuleList([QaptaanDecoderLayer(config, i) for i in range(config.num_hidden_layers)])\n        self.norm = QaptaanRMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_init()\n\n    def forward(\n        self,\n        input_ids: torch.LongTensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        past_key_values: Optional[QaptaanCache] = None,\n        use_cache: Optional[bool] = None,\n    ) -> BaseModelOutputWithPast:\n        if use_cache and (past_key_values is None or not isinstance(past_key_values, QaptaanCache)):\n            past_key_values = QaptaanCache()\n\n        hidden_states = self.embed_tokens(input_ids)\n        for layer in self.layers:\n            hidden_states = layer(\n                hidden_states, attention_mask=attention_mask, past_key_value=past_key_values, use_cache=use_cache\n            )\n        hidden_states = self.norm(hidden_states)\n\n        if use_cache and past_key_values is not None:\n            past_key_values._seen_tokens += input_ids.shape[1]\n\n        return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values)\n\n\nclass QaptaanForCausalLM(QaptaanPreTrainedModel, GenerationMixin):\n    _tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}\n\n    def __init__(self, config: QaptaanConfig):\n        super().__init__(config)\n        self.model = QaptaanModel(config)\n        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)\n        self.post_init()\n\n    def get_input_embeddings(self):\n        return self.model.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.model.embed_tokens = value\n\n    def get_output_embeddings(self):\n        return self.lm_head\n\n    def set_output_embeddings(self, new_embeddings):\n        self.lm_head = new_embeddings\n\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        past_key_values: Optional[QaptaanCache] = None,\n        labels: Optional[torch.LongTensor] = None,\n        use_cache: Optional[bool] = None,\n        **kwargs,\n    ) -> CausalLMOutputWithPast:\n        outputs = self.model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            past_key_values=past_key_values,\n            use_cache=use_cache,\n        )\n        hidden_states = outputs.last_hidden_state\n        logits = self.lm_head(hidden_states)\n\n        loss = None\n        if labels is not None:\n            shift_logits = logits[..., :-1, :].contiguous()\n            shift_labels = labels[..., 1:].contiguous()\n            loss = F.cross_entropy(shift_logits.view(-1, self.config.vocab_size), shift_labels.view(-1))\n\n        return CausalLMOutputWithPast(\n            loss=loss,\n            logits=logits,\n            past_key_values=outputs.past_key_values,\n            hidden_states=outputs.hidden_states,\n        )\n\n    def prepare_inputs_for_generation(\n        self,\n        input_ids,\n        past_key_values=None,\n        attention_mask=None,\n        **kwargs,\n    ):\n        if past_key_values is not None:\n            input_ids = input_ids[:, -1:]\n        return {\n            "input_ids": input_ids,\n            "attention_mask": attention_mask,\n            "past_key_values": past_key_values,\n            "use_cache": True,\n        }\n'
with open(export_dir / 'modeling_qaptaan.py', 'w', encoding='utf-8') as f:
    f.write(modeling_code)
print('✓ Created modeling_qaptaan.py')

# 3. config.json
config_dict = {
    'architectures': ['QaptaanForCausalLM'],
    'model_type': 'qaptaan',
    'auto_map': {
        'AutoConfig': 'configuration_qaptaan.QaptaanConfig',
        'AutoModelForCausalLM': 'modeling_qaptaan.QaptaanForCausalLM'
    },
    'vocab_size': 248320,
    'hidden_size': 1024,
    'intermediate_size': 3584,
    'num_hidden_layers': 24,
    'num_attention_heads': 8,
    'num_key_value_heads': 2,
    'head_dim': 256,
    'rms_norm_eps': 1e-6,
    'tie_word_embeddings': True,
    'max_position_embeddings': 262144,
    'rope_theta': 10000000.0,
    'partial_rotary_factor': 0.25,
    'attn_output_gate': True,
    'full_attention_interval': 4,
    'linear_key_head_dim': 128,
    'linear_value_head_dim': 128,
    'linear_num_key_heads': 16,
    'linear_num_value_heads': 16,
    'linear_conv_kernel_dim': 4,
    'layer_types': ['full_attention' if (i + 1) % 4 == 0 else 'linear_attention' for i in range(24)],
    'use_cache': True,
    'torch_dtype': 'bfloat16'
}
with open(export_dir / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config_dict, f, indent=2)
print('✓ Created config.json')

# 4. generation_config.json
gen_config = {
    'bos_token_id': None,
    'eos_token_id': [151645, 151643],
    'pad_token_id': 151643,
    'do_sample': False,
    'temperature': 0.2,
    'top_p': 0.95,
    'repetition_penalty': 1.05,
    'transformers_version': '4.49.0'
}
with open(export_dir / 'generation_config.json', 'w', encoding='utf-8') as f:
    json.dump(gen_config, f, indent=2)
print('✓ Created generation_config.json')

# 5. Download official tokenizer
from transformers import AutoTokenizer
print('Downloading official Qwen3.5 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3.5-0.8B-Base', trust_remote_code=True)
tokenizer.save_pretrained(str(export_dir))
print('✓ Saved official tokenizer files')

# 6. README.md (Model Card)
model_card = '''---
license: apache-2.0
base_model: kaptaan45/QaptaanLM-0.75B
language:
- en
- code
tags:
- code
- causal-lm
- qwen3.5
- hybrid-attention
- deltanet
- gqa
- instruction-tuning
- sft
- chatml
- kapinstruct
- text-generation
datasets:
- kaptaan45/KapCode-1B
- kaptaan45/KapInstruct-100M
pipeline_tag: text-generation
library_name: transformers
---

# QaptaanLM-0.75B-Instruct: Efficient Hybrid-Attention Code & Reasoning Assistant

[![License](https://img.shields.io/badge/License-Apache%202.0-green.svg)](https://opensource.org/licenses/Apache-2.0)
[![Parameters](https://img.shields.io/badge/Parameters-752M%20(Text--Only)-blue.svg)](#model-specification)
[![Architecture](https://img.shields.io/badge/Architecture-Hybrid%20DeltaNet%20%2B%20GQA-purple.svg)](#architecture)
[![Context Length](https://img.shields.io/badge/Context-256K%20Native-orange.svg)](#model-specification)
[![GitHub](https://img.shields.io/badge/GitHub-QaptaanLM--0.75B-181717.svg?logo=github)](https://github.com/rudy-07/QaptaanLM-0.75B)
[![Kaggle Model](https://img.shields.io/badge/Kaggle-Model-20BEFF.svg?logo=kaggle)](https://www.kaggle.com/models/kaptaan45/qaptaanlm-0.75b)
[![SFT Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20SFT%20Dataset-kaptaan45%2FKapInstruct--100M-orange.svg)](https://huggingface.co/datasets/kaptaan45/KapInstruct-100M)

**QaptaanLM-0.75B-Instruct** is the official instruction-tuned model of the **QaptaanLM-0.75B** family, optimized for Python code generation, bug fixing, SQL query formulation, and multi-turn technical dialogue.

---

## Model Specification

| Property | Value | Notes |
| :--- | :--- | :--- |
| **Model Name** | QaptaanLM-0.75B-Instruct | Text-only instruction-aligned model |
| **Base Architecture** | `Qwen/Qwen3.5-0.8B-Base` | Stripped vision transformer, 100% text capacity |
| **Total Parameters** | **752,382,976 (752M)** | Text-only dense parameters |
| **Hidden Size ($d_{model}$)** | 1024 | Base hidden dimension |
| **Intermediate Size ($d_{ffn}$)** | 3584 | SwiGLU non-linear activation |
| **Total Layers** | 24 | 18 Linear Attention + 6 Full GQA layers (3:1 ratio) |
| **Max Context Window** | 262,144 tokens (256K native) | Interleaved M-RoPE ($\theta = 10,000,000$) |
| **Prompt Format** | Qwen ChatML (`<|im_start|>` / `<|im_end|>`) | Standard system / user / assistant dialogue |
| **Precision** | `bfloat16`, `float16`, `float32` | Hardware accelerated Tensor Cores / TPUs |

---

## Quickstart & Usage

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "kaptaan45/QaptaanLM-0.75B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

messages = [
    {"role": "system", "content": "You are QaptaanLM, an expert programming and reasoning assistant."},
    {"role": "user", "content": "Write a Python function `is_palindrome(s: str) -> bool` that returns True if s is a palindrome."}
]

chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)
```
'''
with open(export_dir / 'README.md', 'w', encoding='utf-8') as f:
    f.write(model_card)
print('✓ Created README.md')


## 5. Validate Local Loading & Inference

In [ ]:
import torch
from transformers import AutoModelForCausalLM

print('Validating model loading from export directory...')
val_model = AutoModelForCausalLM.from_pretrained(
    str(export_dir),
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
val_model.eval()
param_count = sum(p.numel() for p in val_model.parameters())
print(f'✓ Successfully loaded AutoModelForCausalLM! Parameters: {param_count:,} ({param_count/1e6:.2f}M)')

# Test generation
test_prompt = '<|im_start|>user\nWrite a Python function `fibonacci(n: int) -> int` that returns the nth Fibonacci number.<|im_end|>\n<|im_start|>assistant\n'
inputs = tokenizer(test_prompt, return_tensors='pt').to(val_model.device)
with torch.no_grad():
    out = val_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )
print('\nTest Generation Output:\n', tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
del val_model


## 6. Upload Model to Hugging Face Hub

In [ ]:
TARGET_REPO = 'kaptaan45/QaptaanLM-0.75B-Instruct'
print(f'Uploading complete repository to Hugging Face Hub: {TARGET_REPO}...')
api = HfApi(token=HF_TOKEN)
try:
    create_repo(TARGET_REPO, repo_type='model', private=False, token=HF_TOKEN, exist_ok=True)
except Exception as e:
    print('Repo note:', e)

upload_info = api.upload_folder(
    folder_path=str(export_dir),
    repo_id=TARGET_REPO,
    repo_type='model',
    commit_message='feat: publish official QaptaanLM-0.75B-Instruct model (Stage 2 SFT)',
)
print(f'🎉 MODEL SUCCESSFULLY PUBLISHED TO HUGGING FACE: https://huggingface.co/{TARGET_REPO}')
